### Pongartige Spiele mit Asyncio
Bei einem Spiel wie Pong soll z.B. der Ball automatisch alle 0.2 Sekunden bewegt werden.  

Am Einfachsten kann dies mit einem asynchronen While-Loop implementiert werden, der sozusagen nebenbei läuft, ohne
das Notebook zu blockieren.

Eine Funktion, die sozusagen nebenbei läuft, lässt sich mit

```python
async def f():
    ....
```

definieren. Mit
```python
task = asyncio.create_task(countdown())
```
wird diese Funktion zu einen Task gemacht, der nebenbei ausgeführt wird.  
Der Task kann mit `task.cancel()` gestoppt werden.

In [ ]:
import asyncio


async def countdown():
    i = 0
    while True:
        i += 1
        print(i, end=', ')

        await asyncio.sleep(1)  # schlaeft fuer 1 Sekunde ohne zu blockieren


task = asyncio.create_task(countdown())

In [ ]:
task.cancel()  # cancel task

### Schleife mit Flag kontrollieren
Am Einfachsten kontrolliert man einen asynchronen While-Loop mit einer boolschen Variablen (Wahrheitswert).

In [ ]:
import asyncio


running = True


async def countdown():
    i = 0
    while running:
        i += 1
        print(i, end=', ')
        await asyncio.sleep(1)
    print('task completed.')


task = asyncio.create_task(countdown())

In [ ]:
running = False  # task ends

***
Nachstehend ein minimaler Pong-Prototyp.  
Der Ball bewegt sich nur, wenn das Pad mit
ArrowUp oder ArrowDown bewegt wird. Später bewegen wird den Ball dann mit einem asynchronen While-Loop.
***

In [ ]:
import game
import darstellung as D
from ipywidgets import Output
from ipycanvas import MultiCanvas
from IPython.display import display


layout = {'border': '1px solid black'}
out = Output(layout=layout)
mcanvas = MultiCanvas(2, width=game.WIDTH, height=game.HEIGHT, layout=layout)
bg, fg = mcanvas
bg.fill_style = 'blue'
fg.fill_style = 'red'


@out.capture(clear_output=True)
def on_key_down(key, *flags):
    print(key)
    if key == 'n':
        game.new_game()
    elif key == 'ArrowUp':
        game.move_pad(-5)
        game.move_ball()
    elif key == 'ArrowDown':
        game.move_pad(5)
        game.move_ball()


def update(event, **kwargs):
    if event == 'new_game':
        print('redraw everything')
        D.draw_pad(bg, kwargs['pad_y'])
        D.draw_ball(fg, kwargs['ball_pos'])
    if event == 'pad':
        print('redraw pad')
        D.draw_pad(bg, kwargs['pad_y'])
    if event == 'ball':
        print('redraw ball')
        D.draw_ball(fg, kwargs['ball_pos'])
    if event == 'game_over':
        fg.clear()
        fg.fill_text('Game Over!', 30, 100)


game.update = update
game.new_game()

mcanvas.on_key_down(on_key_down)
display(mcanvas, out)

mcanvas.focus()

***
Nun nutzen wir einen asynchronen While-Loop um den Ball alle 0.2 Sekunden zu bewegen.  

```python
state = {'running': False}

async def move_ball():
    while state['running']:
        game.move_ball()
        await asyncio.sleep(0.2)
        
    print('game stopped')
```

Der Loop wird gestartet, wenn die Funktion 
`game.update('new_game', ...)` aufgerufen wird, welche wir mit folgender Funktion überschreiben.

```python
def update(event, **kwargs):
    if event == 'new_game':
        ...
        state['running'] = True
        asyncio.create_task(move_ball())  # Task move_ball wird kreiert und gestartet
```

Der Task stoppt, wenn `state['running']` auf `False` gesetzt wird.  
Das kann durch Drücken von s erreicht werden.

```python
def on_key_down(key, *flags):
    ...
    elif key == 's':
        state['running'] = False
```

***

In [ ]:
import asyncio
import game
import darstellung as D
from ipywidgets import Output
from ipycanvas import MultiCanvas
from IPython.display import display


state = {'running': False}

layout = {'border': '1px solid black'}
out = Output(layout=layout)
mcanvas = MultiCanvas(2, width=game.WIDTH, height=game.HEIGHT, layout=layout)
bg, fg = mcanvas
bg.fill_style = 'blue'
fg.fill_style = 'red'


async def move_ball():  # asynchronen While-Loop, der den Ball alle .2 Sekunden bewegt
    while state['running']:
        game.move_ball()
        await asyncio.sleep(0.2)
    print('game stopped')


@out.capture(clear_output=True)
def on_key_down(key, *flags):
    if key == 'n':
        game.new_game()
    elif key == 's':
        state['running'] = False
    elif key == 'ArrowUp':
        game.move_pad(-5)
    elif key == 'ArrowDown':
        game.move_pad(5)


def update(event, **kwargs):
    if event == 'new_game':
        D.draw_pad(bg, kwargs['pad_y'])
        D.draw_ball(fg, kwargs['ball_pos'])

        state['running'] = True
        asyncio.create_task(move_ball())  # Task move_ball wird kreiert und gestartet
    if event == 'pad':
        D.draw_pad(bg, kwargs['pad_y'])
    if event == 'ball':
        D.draw_ball(fg, kwargs['ball_pos'])
    if event == 'game_over':
        fg.clear()
        fg.fill_text('Game Over!', 30, 50)
        state['running'] = False


game.update = update
game.new_game()

mcanvas.on_key_down(on_key_down)

display(mcanvas, out)
mcanvas.focus()